# Scan2Stage — M3 Room Geometry Normalization
UGScan ZIP/GLB → meters → Z-up → floor Z=0 → clip above 2.5 m → rectangular gallery footprint.


In [ ]:
REPO_URL='https://github.com/6564200/Scan2Stage.git'
REPO_REF='feature/ugscan-zip-glb'
WORKDIR='/content/Scan2Stage'
!rm -rf {WORKDIR}
!git clone -b {REPO_REF} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!bash scripts/colab_bootstrap.sh


## Upload UGScan ZIP or GLB
ZIP is accepted directly. For UGScan GLB the default unit scale is 1.0.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
name=next(iter(uploaded))
assert Path(name).suffix.lower() in {'.zip','.glb','.gltf','.fbx','.obj'}, 'Upload UGScan ZIP/GLB or another supported mesh'
INPUT=Path('/content/Scan2Stage/data')/Path(name).name
INPUT.parent.mkdir(parents=True, exist_ok=True)
Path(name).replace(INPUT)
print(INPUT)


In [ ]:
OUT=Path('/content/Scan2Stage/outputs/m3_room')
!scan2stage {INPUT} --output-dir {OUT} --samples 300000 --source-up y


In [ ]:
import json
report=json.loads((OUT/'report.json').read_text())
room=json.loads((OUT/'room_geometry.json').read_text())
print('Resolved mesh:', report['resolved_mesh'])
print('Format:', report['format'])
print('Bounds in meters:', report['bounds_meters']['extent'])
print('Floor before translation, m:', room['floor']['z_m'])
print('Working volume:', room['working_volume'])
print('Removed above 2.5 m:', room['removed_above_working_height'])
print('Wall candidates:', len(room['wall_candidates']))
print('Rectangle:', room['rectangle'])
room


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fp=np.asarray(room['footprint_xy_m'])
if len(fp):
    closed=np.vstack([fp, fp[0]])
    plt.figure(figsize=(8,8))
    plt.plot(closed[:,0], closed[:,1], '-o')
    plt.axis('equal')
    plt.xlabel('X, m'); plt.ylabel('Y, m'); plt.title('Estimated rectangular gallery footprint')
    plt.grid(True)


In [ ]:
!pytest -q
